# Análise ODBC — `dados_odbc_01-04.xlsx`

Extração **jan–abr/2026** (conforme nome do arquivo). Colunas alinhadas à visão da [[Analise Base Financeira]] (`codcen` / `descen`, `codcdc` / `descdc`, valores, filial); neste export o `lancamento` já vem como data e os valores como número.

Execute a célula abaixo para o resumo exploratório.

In [25]:
# Análise inicial — dados_odbc_01-04.xlsx (ODBC jan–abr/2026)
# Contexto de negócio: ver 00-Zettlelkasten/Analise Base Financeira.md
# (CC 1.x compras / 2.x receitas; descdc COMPRAS/VENDAS DE SUCATAS; filiais G3S vs G&S etc.)

import os
import pandas as pd
from IPython.display import display

_cwd = os.getcwd()
if os.path.isdir(os.path.join(_cwd, "02-Referencias")):
    WORKSPACE = _cwd
elif os.path.isdir(os.path.join(_cwd, "..", "02-Referencias")):
    WORKSPACE = os.path.normpath(os.path.join(_cwd, ".."))
elif os.path.isdir(os.path.join(_cwd, "..", "..", "02-Referencias")):
    WORKSPACE = os.path.normpath(os.path.join(_cwd, "..", ".."))
else:
    raise FileNotFoundError(f"Pasta 02-Referencias não encontrada a partir de: {_cwd}")

PATH_XLSX = os.path.join(WORKSPACE, "02-Referencias", "dados_odbc_01-04.xlsx")

df = pd.read_excel(PATH_XLSX, sheet_name=0)
df["lancamento"] = pd.to_datetime(df["lancamento"])

print("=" * 102)
print("ARQUIVO:", PATH_XLSX)
print("=" * 102)

print(f"Linhas × colunas : {df.shape[0]:,} × {df.shape[1]}")
print(f"Memória (aprox.) : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

ARQUIVO: c:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\dados_odbc_01-04.xlsx
Linhas × colunas : 24,505 × 12
Memória (aprox.) : 15.9 MB


In [27]:
print("\n--- Tipos das colunas e qtd de nulos ---")
info = pd.DataFrame({
    "dtype": df.dtypes.astype(str), 
    "nulos": df.isna().sum()
    })
display(info)


--- Tipos das colunas e qtd de nulos ---


,dtype,nulos
codcen,str,0
descen,str,0
codcdc,str,0
descdc,str,0
lancamento,datetime64[us],0
documento,str,0
nome,str,0
valor_plano,float64,0
valor_centro,float64,0
observacao,str,0


In [17]:
print("\n--- Valores (`valor_plano` / `valor_centro` / `valor_bruto`) ---")
print(f"  Linhas com valor_plano ≠ valor_centro : {(df['valor_plano'] != df['valor_centro']).sum()}")
vb = df["valor_bruto"]
print(f"  valor_bruto — min: R$ {vb.min():,.2f} | max: R$ {vb.max():,.2f}")
print(f"  Soma valor_bruto (todas as linhas)     : R$ {vb.sum():,.2f}")

por_mes = df.groupby(mes, observed=True)["valor_bruto"].agg(["sum", "count"]).rename(
    columns={"sum": "soma_valor_bruto", "count": "qtd"}
)
por_mes["soma_valor_bruto"] = por_mes["soma_valor_bruto"].map(lambda x: f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
print("\n  Soma `valor_bruto` por mês:")
display(por_mes)




--- Valores (`valor_plano` / `valor_centro` / `valor_bruto`) ---
  Linhas com valor_plano ≠ valor_centro : 0
  valor_bruto — min: R$ 0.01 | max: R$ 1,258,158.92
  Soma valor_bruto (todas as linhas)     : R$ 173,565,467.35

  Soma `valor_bruto` por mês:


,soma_valor_bruto,qtd
lancamento,,
2026-01,"30.332.444,61",5604
2026-02,"33.885.894,22",5562
2026-03,"71.105.725,56",7122
2026-04,"38.241.402,96",6217


In [10]:
print("\n--- Período (`lancamento`) ---")
print(f"  Mínimo : {df['lancamento'].min().date()}")
print(f"  Máximo : {df['lancamento'].max().date()}")



mes = df["lancamento"].dt.to_period("M")
print("\n  Lançamentos por mês:")
display(mes.value_counts().sort_index().to_frame("qtd"))


--- Período (`lancamento`) ---
  Mínimo : 2026-01-01
  Máximo : 2026-04-30

  Lançamentos por mês:


,qtd
lancamento,
2026-01,5604
2026-02,5562
2026-03,7122
2026-04,6217


In [11]:
# Dimensões a partir de `descen` (mesmo padrão da base financeira: TIPO / DIVISÃO / CIDADE / DEPT)
partes = df["descen"].astype(str).str.split(" / ", n=3, expand=True)
partes.columns = ["tipo_cc", "divisao_cc", "cidade_cc", "dept_cc"]
df_dim = pd.concat([df[["lancamento", "filial", "valor_bruto", "descdc"]], partes], axis=1)

print("\n--- Filiais (`filial`) — top 10 por |valor_bruto| acumulado ---")
fil = df_dim.groupby("filial", observed=True)["valor_bruto"].apply(lambda s: s.abs().sum()).sort_values(ascending=False)
display(fil.head(10).to_frame("soma_abs_valor_bruto"))




--- Filiais (`filial`) — top 10 por |valor_bruto| acumulado ---


,soma_abs_valor_bruto
filial,
RSE,50514053.87
G3S PRUDENTE,32001096.22
G3S CAMPO GRANDE,29694310.61
G3S MARINGA,22827705.56
G3S DOURADOS,11426796.58
G3S LONDRINA,9921141.80
G&S PRUDENTE,4762286.75
G&S BARUERI,4422755.90
G3S CIDADE ALTA,2325908.10


In [12]:
print("\n--- `divisao_cc` (2ª fatia de `descen`) ---")
display(df_dim["divisao_cc"].value_counts().head(15).to_frame("qtd"))



--- `divisao_cc` (2ª fatia de `descen`) ---


,qtd
divisao_cc,
SELETIVA,20170
EKIPA LOCACOES E SERV G&S,1631
PILARES,577
EKIPA LOCACOES RSE,459
EKIPA SERVICOS - G&S,358
EKIPA,357
BRACOFER,315
EKIPA LOCACOES - RSE,253
TRANSMOVE,109


In [13]:

print("\n--- Categorias (`descdc`) — top 12 por volume (soma |valor_bruto|) ---")
cat = df_dim.groupby("descdc", observed=True)["valor_bruto"].apply(lambda s: s.abs().sum()).sort_values(ascending=False)
display(cat.head(12).to_frame("soma_abs_valor_bruto"))



--- Categorias (`descdc`) — top 12 por volume (soma |valor_bruto|) ---


,soma_abs_valor_bruto
descdc,
VENDAS DE SUCATAS,60273384.78
SEGUROS,30180851.07
COMPRAS DE SUCATAS,30044589.81
TRANSPORTE DE SUCATA,5107447.53
LOCACAO DE MAQUINAS E EQUIPAMENTOS,4313889.70
JUROS E MULTAS,4183751.65
RECEITAS DIVERSAS,3851711.34
MANUTENÇÃO DE VEÍCULOS/MAQUINAS,3342556.23
MAQUINAS E EQUIPAMENTOS,2034722.94


In [14]:

print("\n--- Foco sucata (cf. Analise Base Financeira) ---")
m_compra = df_dim["descdc"].eq("COMPRAS DE SUCATAS")
m_venda = df_dim["descdc"].eq("VENDAS DE SUCATAS")
print(f"  COMPRAS DE SUCATAS : {m_compra.sum():,} linhas | R$ {df_dim.loc[m_compra, 'valor_bruto'].sum():,.2f}")
print(f"  VENDAS DE SUCATAS  : {m_venda.sum():,} linhas | R$ {df_dim.loc[m_venda, 'valor_bruto'].sum():,.2f}")

print("\n--- Amostra (5 linhas aleatórias, seed fixo) ---")
display(
    df.sample(5, random_state=42)[
        ["lancamento", "filial", "codcen", "codcdc", "descdc", "valor_bruto", "documento"]
    ]
)


--- Foco sucata (cf. Analise Base Financeira) ---
  COMPRAS DE SUCATAS : 10,703 linhas | R$ 30,044,589.81
  VENDAS DE SUCATAS  : 3,016 linhas | R$ 60,273,384.78

--- Amostra (5 linhas aleatórias, seed fixo) ---


,lancamento,filial,codcen,codcdc,descdc,valor_bruto,documento
10035,2026-03-16,G3S MARINGA,1.2.4.2,6.1.1,COMPRAS DE SUCATAS,1199.0,BOLC-295753
7502,2026-03-27,G3S MARINGA,1.2.4.2,6.1.1,COMPRAS DE SUCATAS,169.0,BOLC-296898
15667,2026-02-18,G3S CIDADE ALTA,1.2.8.2,6.1.1,COMPRAS DE SUCATAS,372.0,BOLC-293434
5098,2026-04-07,G3S MARINGA,1.2.4.2,6.1.1,COMPRAS DE SUCATAS,168.0,BOLC-297575
10317,2026-03-13,G3S MARINGA,1.2.4.2,6.1.1,COMPRAS DE SUCATAS,117.0,BOLC-295553


In [ ]:
print("--Lancamentos RHGESTOR--")
lanc_rhgestor = df[df["nome"].str.contains("RHGESTOR LTDA", na=False, case=False)]
print(f"Lançamentos com 'RHGESTOR LTDA' na coluna 'nome': {len(lanc_rhgestor)} linhas encontradas.")
display(
    lanc_rhgestor[
        ["lancamento", "filial", "codcen", "codcdc", "descdc", "valor_bruto", "documento", "nome"]
    ]
)

--Lancamentos RHGESTOR--
Lançamentos com 'RHGESTOR LTDA' na coluna 'nome': 22 linhas encontradas.


,lancamento,filial,codcen,codcdc,descdc,valor_bruto,documento,nome
5018,2026-04-08,G3S PRUDENTE,1.1.9,7.5.18,SISTEMAS,3017.86,NFSE-27371,RHGESTOR LTDA
11671,2026-03-09,G3S PRUDENTE,1.2.2.1,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA
11672,2026-03-09,G3S PRUDENTE,1.2.3.1,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA
11673,2026-03-09,G3S PRUDENTE,1.2.4.1,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA
11674,2026-03-09,G3S PRUDENTE,1.2.5.1,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA
11675,2026-03-09,G3S PRUDENTE,1.3.1.1,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA
11676,2026-03-09,G3S PRUDENTE,1.5.1.1,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA
11677,2026-03-09,G3S PRUDENTE,1.4.1.2,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA
11678,2026-03-09,G3S PRUDENTE,1.2.7.1,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA
11679,2026-03-09,G3S PRUDENTE,1.2.8.1,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA


In [23]:
# No ODBC export a coluna de data chama-se `lancamento` (não existe `data`).
df["lancamento"] = pd.to_datetime(df["lancamento"])

df_jan = df[df["lancamento"].dt.month == 1]
df_fev = df[df["lancamento"].dt.month == 2]
df_mar = df[df["lancamento"].dt.month == 3]
df_abril = df[df["lancamento"].dt.month == 4]

In [24]:
lanc_rhgestor_mar = df_mar[df["nome"].str.contains("RHGESTOR LTDA", na=False, case=False)]
lanc_rhgestor_abril = df_abril[df["nome"].str.contains("RHGESTOR LTDA", na=False, case=False)]


display(
    lanc_rhgestor_mar[
        ["lancamento", "filial", "codcen", "codcdc", "descdc", "valor_bruto", "documento", "nome"]
    ]
)

display(
    lanc_rhgestor_abril[
        ["lancamento", "filial", "codcen", "codcdc", "descdc", "valor_bruto", "documento", "nome"]
    ]
)

C:\Users\julio.santana\AppData\Local\Temp\ipykernel_12620\3784662103.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  lanc_rhgestor_mar = df_mar[df["nome"].str.contains("RHGESTOR LTDA", na=False, case=False)]
C:\Users\julio.santana\AppData\Local\Temp\ipykernel_12620\3784662103.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  lanc_rhgestor_abril = df_abril[df["nome"].str.contains("RHGESTOR LTDA", na=False, case=False)]


,lancamento,filial,codcen,codcdc,descdc,valor_bruto,documento,nome
11671,2026-03-09,G3S PRUDENTE,1.2.2.1,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA
11672,2026-03-09,G3S PRUDENTE,1.2.3.1,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA
11673,2026-03-09,G3S PRUDENTE,1.2.4.1,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA
11674,2026-03-09,G3S PRUDENTE,1.2.5.1,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA
11675,2026-03-09,G3S PRUDENTE,1.3.1.1,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA
11676,2026-03-09,G3S PRUDENTE,1.5.1.1,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA
11677,2026-03-09,G3S PRUDENTE,1.4.1.2,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA
11678,2026-03-09,G3S PRUDENTE,1.2.7.1,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA
11679,2026-03-09,G3S PRUDENTE,1.2.8.1,7.5.18,SISTEMAS,3017.86,NFSE-26592,RHGESTOR LTDA


,lancamento,filial,codcen,codcdc,descdc,valor_bruto,documento,nome
5018,2026-04-08,G3S PRUDENTE,1.1.9,7.5.18,SISTEMAS,3017.86,NFSE-27371,RHGESTOR LTDA
